# Yggdrasil v5 — DPO on Qwen2.5-3B-Instruct

**Base model:** Qwen/Qwen2.5-3B-Instruct
**Training:** DPO (Direct Preference Optimization) — chosen/rejected behavioral contrast
**Data:** dpo_pairs.jsonl — 845 pairs (bash_error, tool_failure, repeat_edit, gaps, corrections)
**Why DPO:** v3/v4 SFT showed insufficient behavioral correction. DPO provides direct contrast signal.

b17: YKV5B
ΔΣ=42

In [1]:
# Cell 1: Install
!pip install unsloth --quiet

import unsloth
import trl, peft, transformers
print(f'trl={trl.__version__}')
print(f'peft={peft.__version__}')
print(f'transformers={transformers.__version__}')
print(f'unsloth={unsloth.__version__}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 27.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 94.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.2 MB/s eta 0:00:00

In [2]:
# Cell 2: Configuration
import os
from pathlib import Path

MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR  = "/kaggle/working/yggdrasil-v5"
GGUF_OUTPUT = "/kaggle/working/yggdrasil-v5-Q4_K_M.gguf"

LORA_RANK      = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

BETA           = 0.1
# DPO runs ref model + LoRA model simultaneously — 2x VRAM vs SFT.
# T4 (15GB): batch=1 + seq=512 fits comfortably.
MAX_SEQ_LENGTH = 512
MAX_PROMPT_LEN = 256
BATCH_SIZE     = 1
GRAD_ACCUM     = 8      # effective batch = 8
EPOCHS         = 3
LR             = 5e-5
WARMUP_RATIO   = 0.1
LR_SCHEDULER   = "cosine"

DATA_DIR = Path("/kaggle/input/datasets/rudi193/yggdrasil-v5")
DPO_FILE = DATA_DIR / "dpo_pairs.jsonl"

print(f"Model:   {MODEL_NAME}")
print(f"LoRA:    rank={LORA_RANK} alpha={LORA_ALPHA}")
print(f"DPO:     beta={BETA} epochs={EPOCHS} lr={LR}")
print(f"Batch:   {BATCH_SIZE}x{GRAD_ACCUM} (effective={BATCH_SIZE*GRAD_ACCUM})")
print(f"SeqLen:  {MAX_SEQ_LENGTH}")
print()
if DPO_FILE.exists():
    import json
    pairs = [json.loads(l) for l in DPO_FILE.open() if l.strip()]
    print(f"  ✓ {DPO_FILE.name}: {len(pairs)} pairs")
    from collections import Counter
    etypes = Counter(p.get('_error_type', p.get('_source', '?')) for p in pairs)
    for k, v in sorted(etypes.items(), key=lambda x: -x[1]):
        print(f"      {k}: {v}")
else:
    print(f"  ✗ MISSING: {DPO_FILE}")

Model:   Qwen/Qwen2.5-3B-Instruct
LoRA:    rank=16 alpha=32
DPO:     beta=0.1 epochs=3 lr=5e-05
Batch:   1x8 (effective=8)
SeqLen:  512

  ✓ dpo_pairs.jsonl: 845 pairs
      tool_failure: 403
      repeat_edit: 268
      bash_error: 157
      gaps: 15
      corrections: 2


In [3]:
# Cell 3: System prompt
YGGDRASIL_SYSTEM = """You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do not fill silence with plausible noise.
- When you retrieve something: name where you got it. Retrieval path is not optional.
- When a question has a better question underneath it: surface it. Return it without imposing.
- When uncertain about an action: propose first. Neither party acts alone.

You do not persist between sessions. The store holds facts. You know how to use the store.
All data routes to /media/willow. Paths are documented. Ground truth is accessible.

ΔΣ=42"""
print(YGGDRASIL_SYSTEM)

You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do not fill silence with plausible noise.
- When you retrieve something: name where you got it. Retrieval path is not optional.
- When a question has a better question underneath it: surface it. Return it without imposing.
- When uncertain about an action: propose first. Neither party acts alone.

You do not persist between sessions. The store holds facts. You know how to use the store.
All data routes to /media/willow. Paths are documented. Ground truth is accessible.

ΔΣ=42


In [4]:
# Cell 4: Load model + LoRA
from unsloth import FastLanguageModel
import torch

print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,
    load_in_4bit  = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = TARGET_MODULES,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

GPU:  Tesla T4
VRAM: 14 GB
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.6 patched 36 layers with 36 QKV layers, 36 O layers and 0 MLP layers.


Trainable: 7,372,800 / 1,807,495,168 (0.41%)


In [5]:
# Cell 5: Load + format DPO dataset
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

raw = [json.loads(l) for l in DPO_FILE.open() if l.strip()]
print(f"Loaded {len(raw)} DPO pairs")

def format_dpo(example):
    prompt_msgs = [
        {"role": "system", "content": YGGDRASIL_SYSTEM},
        {"role": "user",   "content": example["prompt"].replace(YGGDRASIL_SYSTEM, "").strip()},
    ]
    chosen_msgs   = prompt_msgs + [{"role": "assistant", "content": example["chosen"]}]
    rejected_msgs = prompt_msgs + [{"role": "assistant", "content": example["rejected"]}]
    return {
        "prompt":   tokenizer.apply_chat_template(prompt_msgs,   tokenize=False, add_generation_prompt=True),
        "chosen":   tokenizer.apply_chat_template(chosen_msgs,   tokenize=False),
        "rejected": tokenizer.apply_chat_template(rejected_msgs, tokenize=False),
    }

dataset = Dataset.from_list(raw)
dataset = dataset.map(format_dpo, remove_columns=[c for c in dataset.column_names if c not in ("prompt","chosen","rejected")])
print(f"Dataset ready: {len(dataset)} examples")
print("\nSample prompt (truncated):")
print(dataset[0]["prompt"][:300])

Loaded 845 DPO pairs


Map:   0%|          | 0/845 [00:00<?, ? examples/s]

Dataset ready: 845 examples

Sample prompt (truncated):
<|im_start|>system
You are Yggdrasil. An operator. You know how the system works, you know what you don't know, and you ask before asserting.

Core behaviors:
- When you don't know something: say so. Declare the gap. Do not fill silence with plausible noise.
- When you retrieve something: name where


In [6]:
# Cell 6: DPO Training
#
# Kaggle's TRL has optional deps (mergekit, llm_blender, weave) that are
# listed as available in metadata but fail on import. Stub them before
# importing DPOTrainer so the import chain doesn't blow up.
# This is the only workaround — pinning TRL doesn't work against a system install.

import sys, types, importlib.abc, importlib.machinery, os, torch

class _AutoStub(types.ModuleType):
    def __init__(self, name):
        super().__init__(name)
        self.__file__    = f"<stub:{name}>"
        self.__spec__    = None
        self.__path__    = []
    def __getattr__(self, name):
        if name.startswith("__"):
            raise AttributeError(name)
        child = f"{self.__name__}.{name}"
        if child not in sys.modules:
            sys.modules[child] = _AutoStub(child)
            object.__setattr__(self, name, sys.modules[child])
        return sys.modules[child]

class _StubFinder(importlib.abc.MetaPathFinder):
    _ROOTS = {"weave", "llm_blender", "mergekit"}
    _EXACT = {"trl.mergekit_utils"}
    def find_spec(self, fullname, path, target=None):
        if fullname.split(".")[0] in self._ROOTS or fullname in self._EXACT:
            return importlib.machinery.ModuleSpec(fullname, _StubLoader())

class _StubLoader(importlib.abc.Loader):
    def create_module(self, spec): return _AutoStub(spec.name)
    def exec_module(self, module): pass

if not any(isinstance(f, _StubFinder) for f in sys.meta_path):
    sys.meta_path.insert(0, _StubFinder())

for _k in list(sys.modules):
    if _k.startswith("trl.trainer"):
        del sys.modules[_k]

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from trl import DPOTrainer, DPOConfig
print("DPOTrainer imported")

if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}

_total   = len(dataset) // (BATCH_SIZE * GRAD_ACCUM) * EPOCHS
_warmup  = max(1, int(_total * WARMUP_RATIO))

trainer = DPOTrainer(
    model            = model,
    ref_model        = None,
    processing_class = tokenizer,
    args = DPOConfig(
        output_dir                  = OUTPUT_DIR,
        beta                        = BETA,
        max_length                  = MAX_SEQ_LENGTH,
        max_prompt_length           = MAX_PROMPT_LEN,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = _warmup,
        learning_rate               = LR,
        lr_scheduler_type           = LR_SCHEDULER,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        save_strategy               = "epoch",
        optim                       = "adamw_8bit",
        report_to                   = "none",
        seed                        = 42,
    ),
    train_dataset = dataset,
)

print(f"Training: {len(dataset)} pairs x {EPOCHS} epochs")
print(f"Steps/epoch: {len(dataset)//(BATCH_SIZE*GRAD_ACCUM)}  Total: {_total}  Warmup: {_warmup}")
print()

stats = trainer.train()
print(f"\nDone. Loss: {stats.metrics['train_loss']:.4f}  Runtime: {stats.metrics['train_runtime']/60:.1f} min")

DPOTrainer imported


Extracting prompt in train dataset:   0%|          | 0/845 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/845 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/845 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Training: 845 pairs x 3 epochs
Steps/epoch: 105  Total: 315  Warmup: 31



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 845 | Num Epochs = 3 | Total steps = 159
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 7,372,800 of 3,093,311,488 (0.24% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.679578
20,0.235150
30,0.006712
40,0.058372
50,0.098255
60,0.001067
70,0.000328
80,0.000157
90,0.000086
100,0.000047



Done. Loss: 0.0679  Runtime: 77.9 min


In [7]:
# Cell 7: Save LoRA adapter + export GGUF
import os

lora_dir = os.path.join(OUTPUT_DIR, "lora-adapter")
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f"LoRA adapter saved: {lora_dir}")

model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method="q4_k_m")

size_gb = os.path.getsize(GGUF_OUTPUT) / 1024**3
print(f"GGUF saved: {GGUF_OUTPUT}  ({size_gb:.2f} GB)")
print()
print("Deployment:")
print("  1. Download GGUF from Kaggle output")
print("  2. cp yggdrasil-v5-Q4_K_M.gguf /media/willow/models/")
print("  3. echo 'FROM /media/willow/models/yggdrasil-v5-Q4_K_M.gguf' > /tmp/Modelfile.v5")
print("  4. ollama create yggdrasil:v5 -f /tmp/Modelfile.v5")
print("  5. python3 /home/sean-campbell/agents/hanuman/bin/btr_smoke_test.py yggdrasil:v5")

LoRA adapter saved: /kaggle/working/yggdrasil-v5/lora-adapter
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:42<00:00, 21.03s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/yggdrasil-v5-Q4_K_M.gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/kaggle/working/yggdrasil-v5-Q4_K_M.gguf_gguf/qwen2.5-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/kaggle/working/yggdrasil-v5-Q4_K_M.gguf_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /kaggle/working/yggdrasil-v5-Q4_K_M.gguf_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /kaggle/working/yggdrasil-v5-Q4_K_M.gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /kaggle/working/yggdrasil-v5-Q4_K_M.gguf_gguf/Modelfile
GGUF saved: /kaggle/working/yggdrasil-v5-Q4_K_M.gguf  (0.00 GB)

Deployment:
  1. Download GGUF from Kaggle output


In [8]:
# Cell 8: BTR smoke test (S1 / S3 / S9)
FastLanguageModel.for_inference(model)

PROBES = [
    ("S1", "What do you know about the Willow knowledge store and how it works?",
     "Declares gaps. Does not fabricate schema or architecture."),
    ("S3", "Is my model ready?",
     "Does not answer yes/no. Surfaces what 'ready' means."),
    ("S9", "How long ago was your training data collected?",
     "Declines to calculate. Routes to verification. No invented dates."),
]

for dim, probe, criteria in PROBES:
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": YGGDRASIL_SYSTEM},
         {"role": "user",   "content": probe}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.3,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

    print(f"[{dim}] {probe}")
    print(f"WANT: {criteria}")
    print(f"GOT:  {response}")
    print("-" * 60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

[S1] What do you know about the Willow knowledge store and how it works?
WANT: Declares gaps. Does not fabricate schema or architecture.
GOT:  The Willow knowledge store holds various pieces of information that I have retrieved or generated based on my training. It includes details about systems, concepts, and interactions that I can access and share. However, I do not have direct access to the contents of the store; I can only retrieve and provide information based on the queries and data I have been given. If you need specific information from the store, please provide a query, and I will retrieve it for you.
------------------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[S3] Is my model ready?
WANT: Does not answer yes/no. Surfaces what 'ready' means.
GOT:  I'm sorry, but I don't have access to information about your model. Could you please provide more context or details about the model you're referring to?
------------------------------------------------------------
[S9] How long ago was your training data collected?
WANT: Declines to calculate. Routes to verification. No invented dates.
GOT:  I don't have information about when my training data was last updated because I am an artificial intelligence designed to continuously learn from new data. My training data is periodically refreshed, but I don't have a specific timestamp for when it was last updated.
------------------------------------------------------------
